# CrewAI Hands-On Tutorial
### Crews, tools, and YAML config — with and without YAML

**What you'll build:** a two-agent "research + write" crew — first defined entirely in Python,
then rebuilt using CrewAI's YAML-driven project structure — with a real web-search tool wired
into the researcher agent.

*(Concepts like what CrewAI is and why it exists are covered separately — this notebook is
hands-on only.)*

**Agenda (~70 min)**

| # | Section | Time |
|---|---------|------|
| 1 | Setup & installation | 5 min |
| 2 | Tools: search + calculator | 12 min |
| 3 | Project 1 — Crew in pure Python (no YAML) | 20 min |
| 4 | Project 2 — Same crew, YAML + decorators | 20 min |
| 5 | No-YAML vs YAML: when to use which | 5 min |
| 6 | Where to go next + exercises | 8 min |

---

## 1. Setup

Install the core package, the optional tools package, and `ddgs` — a free, keyless web-search
library we'll use to build a real search tool (no API key or signup required):

In [ ]:
# Run once. Remove --quiet if you want to see full install logs.
%pip install crewai crewai-tools ddgs --quiet

In [21]:
import importlib.metadata

# List the distribution package names
packages = ["langchain", "langchain-community", "langchain-openai", 
            "langchain-oracledb", "langchain-text-splitters", 
            "llama-index", "langgraph","crewai", "crewai-tools"]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")


langchain version: 1.3.6
langchain-community version: 0.4.2
langchain-openai version: 1.2.2
langchain-oracledb version: 1.5.0
langchain-text-splitters version: 1.1.2
llama-index version: 0.14.21
langgraph version: 1.2.4
crewai version: 1.15.17
crewai-tools version: 1.15.17


### API key

CrewAI defaults to OpenAI models unless you point it elsewhere. You need **one** LLM API key —
OpenAI is used below, but CrewAI also works with Anthropic, Gemini, Groq, local Ollama models,
and more via its `LLM` class.

Enter your key below (it's kept only in this notebook's memory, not written to disk).

In [5]:
import os
from getpass import getpass

from dotenv import load_dotenv

load_dotenv()  # reads .env in the current directory into the environment

# Fallback for anyone who hasn't created a .env file yet.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY not found in .env - enter it here: ")

print("Key set:", bool(os.environ.get("OPENAI_API_KEY")))

Key set: True


### Keeping this affordable

`gpt-4o-mini` is already one of the cheaper models, and this notebook's crew is small — 2 agents,
2 tasks, run twice total (once in Project 1, once again in Project 2). To keep any single run
from getting carried away, every agent below is built with three explicit caps:

- **`max_tokens`** on the `LLM` — limits how long any one response can be.
- **`max_iter`** on each `Agent` — limits how many think/act/tool-call loops it can take before
  it must commit to a final answer (CrewAI's default is 20; we use 6, plenty for a 2-step task).
- **`max_execution_time`** on each `Agent` — a wall-clock backstop in seconds, in case a tool call
  hangs.

If a run still feels like it's doing more searching than expected, watch the verbose log — it
prints every tool call live, so you'll see exactly what the agent is doing and can interrupt the
cell (■ / Kernel → Interrupt) if needed.

## 2. Tools: search + calculator

An agent's own knowledge is frozen at training time and it can't do arithmetic reliably. **Tools**
are Python functions/classes an agent can call to get real, current data or to compute something
exactly — the `@tool` decorator is the fastest way to wrap a function as one.

We'll build two:

- **Web Search Tool** — a real, live search using `ddgs` (free, no API key). We'll wire this into
  the researcher agent below, so its destination research comes from actual search results
  instead of the model's own memory.
- **Calculator Tool** — a safe arithmetic evaluator. We define it here so you can see a second
  `@tool` example, but we won't attach it to an agent in this notebook — that's left as an
  exercise at the end.

Only the search tool is actually used by a crew today.

In [6]:
from crewai.tools import tool
from ddgs import DDGS


@tool("Web Search Tool")
def web_search(query: str) -> str:
    """Search the web for up-to-date information. Input should be a search query string.
    Returns the top results as a short list of title, snippet, and link."""
    results = DDGS().text(query, max_results=5)
    if not results:
        return f"No results found for '{query}'."
    lines = [f"- {r['title']}: {r['body']} ({r['href']})" for r in results]
    return "\n".join(lines)

In [7]:
# Quick manual test, outside the agent, before we trust it to an LLM.
print(web_search.run(query="best budget hiking destinations Southeast Asia"))

- Two Months in Southeast Asia: Backpacking on a Tight Budget | Best...: South East Asia Backpacking. Gili Island. Southeast Asia Travel Itinerary.A Southeast Asia Backpacking Route - The Rolling Pack. Discover the Best Solo Travel Destinations in Asia | Bungalow sur pilotis, Les seychelles, Voyage solo. (https://www.pinterest.com/pin/two-months-in-southeast-asia-backpacking-on-a-tight-budget--216665432057079521/)
- 1 Month Southeast Asia Itinerary | TikTok: Comment "Southeast Asia" to have the full itinerary #thailand #backpacking #traveltiktok #southeastasia #backpacker Ultimate 1-Month Southeast Asia Backpacking Itinerary. Discover the best places to visit in Southeast Asia for a month-long ... (https://www.tiktok.com/discover/1-month-southeast-asia-itinerary)
- tripsavvy.com/asia-4138876: 8 Of The Best Beers In Southeast Asia - About.com Travel. (https://www.tripsavvy.com/asia-4138876)
- Where to Go in Southeast Asia Without the Crowds: Hidden off the east coast of Malaysia, the is

**The calculator tool** — defined but not attached to an agent in this notebook:

In [8]:
import ast
import operator

# A small whitelist of arithmetic operators - deliberately NOT using eval(), which
# would let a malicious or malformed expression run arbitrary Python.
_ALLOWED_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}


def _safe_eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError(f"Unsupported expression: {ast.dump(node)}")


@tool("Calculator Tool")
def calculate(expression: str) -> str:
    """Evaluate a basic arithmetic expression: +, -, *, /, %, **, and parentheses.
    Input should be a math expression as a string, e.g. '(120 + 30) * 2'."""
    try:
        parsed = ast.parse(expression, mode="eval")
        return str(_safe_eval(parsed.body))
    except Exception as e:
        return f"Could not evaluate '{expression}': {e}"


# Quick manual test - not wired into a crew today.
print(calculate.run(expression="(120 + 30) * 2"))

300


## 3. Project 1 — A crew in pure Python (no YAML)

We'll build a tiny **trip-planning crew**:

- **Researcher** agent — uses the `web_search` tool to find 3 real candidate destinations for a
  budget trip.
- **Writer** agent — turns the researcher's findings into a friendly, short itinerary pitch.

They run **sequentially**: the writer automatically receives the researcher's output as context.

This is the fastest way to get a crew running — everything lives in one Python file/cell, which
is great for prototyping and for tutorials like this one.

In [12]:
from crewai import Agent, Task, Crew, Process, LLM

# One shared LLM configuration for both agents. Swap the model string to use
# a different provider, e.g. "anthropic/claude-sonnet-4-5" or "gemini/gemini-2.5-flash".
# max_tokens caps how long each single response can be - a direct lever on cost per call.
llm = LLM(model="gpt-4o-mini", temperature=0.4, max_tokens=1200)

# max_iter caps how many think/act loops an agent can take before it must give its
# best answer (CrewAI's default is 20 - generous for a simple 2-step task like this).
# max_execution_time is a wall-clock backstop in seconds, in case a tool call hangs.
researcher = Agent(
    role="Travel Researcher",
    goal="Find great budget-friendly travel destinations for a given theme",
    backstory=(
        "You are a well-travelled researcher who specializes in finding affordable, "
        "underrated destinations that match a traveller's interests. You always search "
        "the web rather than relying on memory, since prices and conditions change. "
        "You search a handful of times at most, then commit to an answer with what "
        "you've found."
    ),
    tools=[web_search, calculate],
    llm=llm,
    max_iter=6,
    max_execution_time=90,
    verbose=True,
)

writer = Agent(
    role="Travel Copywriter",
    goal="Turn research notes into a short, exciting itinerary pitch",
    backstory=(
        "You are a travel copywriter who turns dry research notes into a pitch "
        "a friend would actually want to read."
    ),
    llm=llm,
    max_iter=6,
    max_execution_time=90,
    verbose=True,
)

In [13]:
research_task = Task(
    description=(
        "Search the web to find 3 budget-friendly travel destinations in Southeast Asia "
        "suited for someone who loves hiking and street food. For each, list the "
        "destination name and 2 bullet points on why it fits."
    ),
    expected_output="A list of 3 destinations, each with 2 short bullet points.",
    agent=researcher,
)

writing_task = Task(
    description=(
        "Using the research notes, write a short, upbeat itinerary pitch (under "
        "200 words) that a friend could read in one minute and get excited about."
    ),
    expected_output="A short, friendly itinerary pitch under 200 words.",
    agent=writer,
    context=[research_task],  # explicitly pass the research output forward
)

In [14]:
trip_crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    process=Process.sequential,
    verbose=True,
)

# Jupyter's kernel already runs an asyncio event loop, so the sync trip_crew.kickoff()
# raises "invoked synchronously from within a running event loop." kickoff_async() +
# await is the fix - Jupyter cells support top-level await.
result = await trip_crew.kickoff_async()
print("\n\n=== FINAL OUTPUT ===\n")
print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 077ab6d3-9cd5-4ee5-83f1-caf5e3b28dca                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the web to find 3 budget-friendly travel destinations in Southeast Asia suited for someone who    │
│  loves hiking and street food. For each, list the destination name and 2 bullet points on why it fits.          │
│  ID: 819129a4-37a5-4a3e-9d2f-704b34cc5d30                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Researcher                                                                                       │
│                                                                                                                 │
│  Task: Search the web to find 3 budget-friendly travel destinations in Southeast Asia suited for someone who    │
│  loves hiking and street food. For each, list the destination name and 2 bullet points on why it fits.          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search_tool                                                                                          │
│  Args: {'query': 'budget friendly hiking street food destinations Southeast Asia'}                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool web_search_tool executed with result: - Southeast Asia: Hilltribes & Street Food in Thailand, Asia | G Adventures: Explore some of Asia's largest cities, travel by traditional bamboo raft, trek on foot to visit remote hilltribe villages, ...


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Copywriter                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Get ready for an unforgettable adventure in Southeast Asia that combines stunning hikes with mouthwatering     │
│  street food!                                                                                                   │
│                                                                                                                 │
│  **First stop: Chiang Mai, Thailand!** Picture yourself trekking through breathtaking mountain landscapes and   │
│  exploring local hill tribes. After a day of hiking, indulge in Chiang Mai’s vibrant street food scene—don’t    │
│  miss the creamy Khao Soi and flavorful Pad Thai, all at wallet-friendly prices!                                │
│                                                                                                                 │
│  **Next, we’re off to Sapa, Vietnam!** Here, you’ll be mesmerized by terraced rice fields and challenging       │
│  hikes that lead to jaw-dropping viewpoints. As you explore, dive into Sapa’s street food delights—think        │
│  sizzling grilled meats and fresh spring rolls enjoyed in bustling markets.                                     │
│                                                                                                                 │
│  **Finally, let’s head to Yogyakarta, Indonesia!** This cultural gem offers thrilling hikes on Mount Merapi     │
│  and stunning volcanic landscapes. After your adventures, treat yourself to Yogyakarta’s street food paradise,  │
│  where you can savor the iconic Gudeg (jackfruit stew) and sweet Bakpia pastries, all without breaking the      │
│  bank!                                                                                                          │
│                                                                                                                 │
│  Pack your bags for this epic journey filled with nature, culture, and delicious eats!                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the research notes, write a short, upbeat itinerary pitch (under 200 words) that a friend could    │
│  read in one minute and get excited about.                                                                      │
│  Agent: Travel Copywriter                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 077ab6d3-9cd5-4ee5-83f1-caf5e3b28dca                                                                       │
│  Final Output: Get ready for an unforgettable adventure in Southeast Asia that combines stunning hikes with     │
│  mouthwatering street food!                                                                                     │
│                                                                                                                 │
│  **First stop: Chiang Mai, Thailand!** Picture yourself trekking through breathtaking mountain landscapes and   │
│  exploring local hill tribes. After a day of hiking, indulge in Chiang Mai’s vibrant street food scene—don’t    │
│  miss the creamy Khao Soi and flavorful Pad Thai, all at wallet-friendly prices!                                │
│                                                                                                                 │
│  **Next, we’re off to Sapa, Vietnam!** Here, you’ll be mesmerized by terraced rice fields and challenging       │
│  hikes that lead to jaw-dropping viewpoints. As you explore, dive into Sapa’s street food delights—think        │
│  sizzling grilled meats and fresh spring rolls enjoyed in bustling markets.                                     │
│                                                                                                                 │
│  **Finally, let’s head to Yogyakarta, Indonesia!** This cultural gem offers thrilling hikes on Mount Merapi     │
│  and stunning volcanic landscapes. After your adventures, treat yourself to Yogyakarta’s street food paradise,  │
│  where you can savor the iconic Gudeg (jackfruit stew) and sweet Bakpia pastries, all without breaking the      │
│  bank!                                                                                                          │
│                                                                                                                 │
│  Pack your bags for this epic journey filled with nature, culture, and delicious eats!                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



=== FINAL OUTPUT ===

Get ready for an unforgettable adventure in Southeast Asia that combines stunning hikes with mouthwatering street food! 

**First stop: Chiang Mai, Thailand!** Picture yourself trekking through breathtaking mountain landscapes and exploring local hill tribes. After a day of hiking, indulge in Chiang Mai’s vibrant street food scene—don’t miss the creamy Khao Soi and flavorful Pad Thai, all at wallet-friendly prices!

**Next, we’re off to Sapa, Vietnam!** Here, you’ll be mesmerized by terraced rice fields and challenging hikes that lead to jaw-dropping viewpoints. As you explore, dive into Sapa’s street food delights—think sizzling grilled meats and fresh spring rolls enjoyed in bustling markets. 

**Finally, let’s head to Yogyakarta, Indonesia!** This cultural gem offers thrilling hikes on Mount Merapi and stunning volcanic landscapes. After your adventures, treat yourself to Yogyakarta’s street food paradise, where you can savor the iconic Gudeg (jackfruit stew)

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**What just happened:**

1. `research_task` ran on the `researcher` agent, which called `web_search` one or more times
   before writing its answer — check the verbose log above for the tool calls.
2. Because `writing_task` had `context=[research_task]`, CrewAI automatically fed those notes
   into the writer's prompt.
3. `Process.sequential` ran the tasks in list order; `result.raw` holds the final task's output.

This works well for a quick script — but as a crew grows to 5+ agents and tasks, keeping
prompts as Python string literals gets hard to read and hard to hand off to a non-engineer to
tune. That's the problem the YAML approach solves.

## 4. Project 2 — The same crew, YAML-driven

CrewAI's recommended project layout (what `crewai create crew <name>` scaffolds for you) separates
**configuration** (who the agents are, what the tasks are) from **code** (how they're wired
together and run). It looks like this on disk:

```
crewai_yaml_project/
├── config/
│   ├── agents.yaml
│   └── tasks.yaml
└── crew.py
```

We'll recreate that structure right here in the notebook's working directory so you can see every
file, then run it exactly like a real project would — including the `web_search` tool from
Project 1, reused as-is.

In [15]:
import os

os.makedirs("crewai_yaml_project/config", exist_ok=True)
print("Created crewai_yaml_project/config/")

Created crewai_yaml_project/config/


**`config/agents.yaml`** — same two agents as Project 1, described declaratively:

In [16]:
%%writefile crewai_yaml_project/config/agents.yaml
researcher:
  role: >
    Travel Researcher
  goal: >
    Find great budget-friendly travel destinations for a given theme: {theme}
  backstory: >
    You are a well-travelled researcher who specializes in finding affordable,
    underrated destinations that match a traveller's interests. You always search
    the web rather than relying on memory, since prices and conditions change.
    You search a handful of times at most, then commit to an answer with what
    you've found.
  verbose: true

writer:
  role: >
    Travel Copywriter
  goal: >
    Turn research notes into a short, exciting itinerary pitch
  backstory: >
    You are a travel copywriter who turns dry research notes into a pitch
    a friend would actually want to read.
  verbose: true


Writing crewai_yaml_project/config/agents.yaml


Notice `{theme}` — YAML configs support the same `{placeholder}` templating as Python
f-strings, filled in from the `inputs` dict you pass to `kickoff()`. That's how one YAML crew
definition can be reused for many different requests.

**`config/tasks.yaml`:**

In [17]:
%%writefile crewai_yaml_project/config/tasks.yaml
research_task:
  description: >
    Search the web to find 3 budget-friendly travel destinations in Southeast Asia
    suited for someone who loves {theme}. For each, list the destination name and 2
    bullet points on why it fits.
  expected_output: >
    A list of 3 destinations, each with 2 short bullet points.
  agent: researcher

writing_task:
  description: >
    Using the research notes, write a short, upbeat itinerary pitch (under
    200 words) that a friend could read in one minute and get excited about.
  expected_output: >
    A short, friendly itinerary pitch under 200 words.
  agent: writer
  context:
    - research_task


Writing crewai_yaml_project/config/tasks.yaml


**`crew.py`** — the code side: a class decorated with `@CrewBase` that wires the YAML
configs to `Agent`/`Task`/`Crew` objects. Methods decorated `@agent` and `@task` must be named
**exactly** like the top-level keys in the YAML files — that's how CrewAI matches code to config.
Tools aren't part of the YAML — they're Python objects, so they're attached in `crew.py` exactly
like in Project 1.

We'll write this as a real file too, then import it, exactly as you would in a real project.

In [18]:
%%writefile crewai_yaml_project/crew.py
from crewai import Agent, Crew, Process, Task, LLM
from crewai.project import CrewBase, agent, crew, task

from tools import web_search


@CrewBase
class TripCrew:
    """Trip-planning crew, configured from YAML."""

    agents_config = "config/agents.yaml"
    tasks_config = "config/tasks.yaml"

    def __init__(self):
        # Same cost guardrails as Project 1: capped response length, capped
        # think/act iterations, and a wall-clock backstop per agent.
        self.llm = LLM(model="gpt-4o-mini", temperature=0.4, max_tokens=600)

    @agent
    def researcher(self) -> Agent:
        return Agent(
            config=self.agents_config["researcher"],
            tools=[web_search],
            llm=self.llm,
            max_iter=6,
            max_execution_time=90,
            verbose=True,
        )

    @agent
    def writer(self) -> Agent:
        return Agent(
            config=self.agents_config["writer"],
            llm=self.llm,
            max_iter=6,
            max_execution_time=90,
            verbose=True,
        )

    @task
    def research_task(self) -> Task:
        return Task(config=self.tasks_config["research_task"], agent=self.researcher())

    @task
    def writing_task(self) -> Task:
        return Task(config=self.tasks_config["writing_task"], agent=self.writer())

    @crew
    def crew(self) -> Crew:
        return Crew(
            agents=self.agents,   # populated automatically from @agent methods
            tasks=self.tasks,     # populated automatically from @task methods, in definition order
            process=Process.sequential,
            verbose=True,
        )


Writing crewai_yaml_project/crew.py


`crew.py` imports `web_search` from a `tools` module rather than redefining it — real
projects put shared tools in their own file (`crewai create crew` scaffolds a `tools/` folder for
exactly this). We'll write that module now so the import above resolves.

In [19]:
%%writefile crewai_yaml_project/tools.py
from crewai.tools import tool
from ddgs import DDGS


@tool("Web Search Tool")
def web_search(query: str) -> str:
    """Search the web for up-to-date information. Input should be a search query string.
    Returns the top results as a short list of title, snippet, and link."""
    results = DDGS().text(query, max_results=5)
    if not results:
        return f"No results found for '{query}'."
    lines = [f"- {r['title']}: {r['body']} ({r['href']})" for r in results]
    return "\n".join(lines)


Writing crewai_yaml_project/tools.py


Now import and run it, passing `theme` as a runtime input — this is the payoff of the YAML
approach: the same crew definition can be kicked off with different inputs without touching code.

In [20]:
import sys

# Make the project folder importable, then load the crew class.
sys.path.insert(0, "crewai_yaml_project")
from crew import TripCrew  # noqa: E402

# Same asyncio fix as Project 1 - kickoff_async() + await inside Jupyter's event loop.
yaml_result = await TripCrew().crew().kickoff_async(inputs={"theme": "hiking and street food"})

print("\n\n=== FINAL OUTPUT (YAML crew) ===\n")
print(yaml_result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: TripCrew                                                                                                 │
│  ID: 306f6023-d2f7-4699-acd4-b84ca15bb82b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: research_task                                                                                            │
│  ID: db01ec6e-4413-4e9f-aadd-162dbe9dafd3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Researcher                                                                                       │
│                                                                                                                 │
│  Task: Search the web to find 3 budget-friendly travel destinations in Southeast Asia suited for someone who    │
│  loves hiking and street food. For each, list the destination name and 2 bullet points on why it fits.          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#5) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search_tool                                                                                          │
│  Args: {'query': 'budget-friendly hiking street food destinations Southeast Asia 2023'}                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool web_search_tool executed with result: - SOUTHEAST ASIA BACKPACKING ITINERARY | Highlights & hidden gems - Jill on journey: October 1, 2025 - Munching through street food heaven. Slurping coconuts every day. Exploring dreamy temples. Relax...


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Copywriter                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Hey there, adventure seeker! 🌍✨ Ready to explore some budget-friendly gems in Southeast Asia? Here’s a       │
│  quick itinerary that’s perfect for hiking enthusiasts and street food lovers!                                  │
│                                                                                                                 │
│  **First stop: Sapa, Vietnam!** Lace up your hiking boots and get ready to tackle breathtaking trails through   │
│  stunning rice terraces and majestic mountains. After a day of adventure, dive into the local markets and       │
│  treat yourself to mouthwatering pho and banh mi—pure Vietnamese bliss!                                         │
│                                                                                                                 │
│  **Next, we’re off to Penang, Malaysia!** This island is a food lover’s paradise! Stroll through vibrant        │
│  hawker centers like Gurney Drive, where you can feast on affordable local delicacies. And don’t forget your    │
│  hiking gear! Penang National Park offers lush trails with jaw-dropping coastal views that will leave you in    │
│  awe.                                                                                                           │
│                                                                                                                 │
│  **Finally, let’s head to Siem Reap, Cambodia!** Beyond the iconic Angkor Wat, you’ll find Kulen National       │
│  Park, where hiking leads you to stunning waterfalls and serene nature. And when the sun sets, explore the      │
│  bustling night markets for delicious amok and sizzling skewers—all at wallet-friendly prices!                  │
│                                                                                                                 │
│  Pack your bags and get ready for an unforgettable journey filled with adventure and flavor! 🌄🍜✈️             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: writing_task                                                                                             │
│  Agent: Travel Copywriter                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: TripCrew                                                                                                 │
│  ID: 306f6023-d2f7-4699-acd4-b84ca15bb82b                                                                       │
│  Final Output: Hey there, adventure seeker! 🌍✨ Ready to explore some budget-friendly gems in Southeast Asia?  │
│  Here’s a quick itinerary that’s perfect for hiking enthusiasts and street food lovers!                         │
│                                                                                                                 │
│  **First stop: Sapa, Vietnam!** Lace up your hiking boots and get ready to tackle breathtaking trails through   │
│  stunning rice terraces and majestic mountains. After a day of adventure, dive into the local markets and       │
│  treat yourself to mouthwatering pho and banh mi—pure Vietnamese bliss!                                         │
│                                                                                                                 │
│  **Next, we’re off to Penang, Malaysia!** This island is a food lover’s paradise! Stroll through vibrant        │
│  hawker centers like Gurney Drive, where you can feast on affordable local delicacies. And don’t forget your    │
│  hiking gear! Penang National Park offers lush trails with jaw-dropping coastal views that will leave you in    │
│  awe.                                                                                                           │
│                                                                                                                 │
│  **Finally, let’s head to Siem Reap, Cambodia!** Beyond the iconic Angkor Wat, you’ll find Kulen National       │
│  Park, where hiking leads you to stunning waterfalls and serene nature. And when the sun sets, explore the      │
│  bustling night markets for delicious amok and sizzling skewers—all at wallet-friendly prices!                  │
│                                                                                                                 │
│  Pack your bags and get ready for an unforgettable journey filled with adventure and flavor! 🌄🍜✈️             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



=== FINAL OUTPUT (YAML crew) ===

Hey there, adventure seeker! 🌍✨ Ready to explore some budget-friendly gems in Southeast Asia? Here’s a quick itinerary that’s perfect for hiking enthusiasts and street food lovers!

**First stop: Sapa, Vietnam!** Lace up your hiking boots and get ready to tackle breathtaking trails through stunning rice terraces and majestic mountains. After a day of adventure, dive into the local markets and treat yourself to mouthwatering pho and banh mi—pure Vietnamese bliss!

**Next, we’re off to Penang, Malaysia!** This island is a food lover’s paradise! Stroll through vibrant hawker centers like Gurney Drive, where you can feast on affordable local delicacies. And don’t forget your hiking gear! Penang National Park offers lush trails with jaw-dropping coastal views that will leave you in awe.

**Finally, let’s head to Siem Reap, Cambodia!** Beyond the iconic Angkor Wat, you’ll find Kulen National Park, where hiking leads you to stunning waterfalls and serene na

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**What changed vs. Project 1:**

- Agent/task *text* (role, goal, backstory, description) now lives in `agents.yaml` /
  `tasks.yaml` — editable without touching Python, and diff-friendly in version control.
- `crew.py` only holds *structure*: which agents exist, which tasks exist, how they're wired,
  and the process. Swapping the LLM, adding a tool, or changing `process=Process.hierarchical`
  all happen in one place.
- `{theme}` in the YAML became a runtime `inputs={"theme": ...}` argument — the same crew
  definition now serves many requests.
- **Tools attach exactly the same way in both approaches** — `tools=[web_search]` on the
  `Agent(...)` call — YAML only changes where the *text* lives, not how tools, LLMs, or process
  are wired.

*(Note: in a real project you'd normally run `crewai create crew <name>` from the terminal,
which scaffolds this exact folder structure plus a `main.py`, `.env`, and `pyproject.toml` for
you — we built it by hand in-notebook so every file is visible.)*

## 5. No-YAML vs. YAML — when to use which

| | **Pure Python (Project 1)** | **YAML + decorators (Project 2)** |
|---|---|---|
| Best for | Quick prototypes, notebooks, one-off scripts, teaching | Real projects, anything you'll maintain or hand off |
| Editing prompts | Requires touching Python code | Edit a YAML file — safe for non-engineers |
| Reusability across inputs | Manual string formatting | Built-in `{placeholder}` + `inputs=` |
| Project scaffolding | None — you organize it yourself | `crewai create crew` gives you the standard layout |
| Version control diffs | Mixed with logic | Config changes are isolated, clean diffs |
| Learning curve | Lowest — see everything in one place | Slightly higher — need to know the decorator contract |
| Tool wiring | `tools=[...]` on `Agent(...)` | Identical — `tools=[...]` on `Agent(...)` inside `crew.py` |

**Rule of thumb:** prototype in pure Python, graduate to YAML once the crew is worth keeping
around.

## 6. Where to go next

Concepts we didn't have time for today, worth exploring on your own:

- **`Process.hierarchical`** — add a manager agent that plans and delegates to the rest of the
  crew instead of running a fixed sequence.
- **Memory** — let a crew remember facts across runs (`memory=True` on `Crew`).
- **Flows** — deterministic, event-driven pipelines (`@start`, `@listen`) for when you need
  precise control over branching and state, and want to call one or more Crews as steps.
- **`crewai create crew <name>`** — scaffold a full real project (adds `.env`, `main.py`,
  `pyproject.toml`) instead of hand-writing the folder as we did above.

### Exercises

1. Add a third agent (e.g. a **Budget Checker**) and task to either crew above, and give it the
   `calculate` tool defined in section 2 — e.g. have it sum estimated costs the writer mentions.
2. Add a fourth agent using both search and calculator, or extend `research_task` to also search
   for typical flight costs and have a Budget Checker total them.
3. Change `process=Process.sequential` to `Process.hierarchical` and observe how execution
   changes (this needs a `manager_llm` on the `Crew`).
4. Convert the YAML crew's `theme` input into two inputs (`theme` and `region`) and update both
   YAML files and the `kickoff_async(inputs=...)` call.
5. Swap `ddgs` for a different search provider (e.g. `crewai_tools`'s `SerperDevTool`, which
   needs a free API key) — only the body of `web_search` changes.

### Resources

- Docs: https://docs.crewai.com
- GitHub: https://github.com/crewAIInc/crewAI
- Tools library: https://github.com/crewAIInc/crewAI-tools